# Notebook 3 — Query Routing: Gemini Flash Classifier & Cypher Generator
**Layer:** Retrieval routing · **Scope:** Natural language → intent type + Cypher query  
**Inputs:** User query string · Neo4j (from Notebook 1)  
**Outputs:** `intent_type` · `slots` JSON · parameterised Cypher result

## 3.1 Install & import dependencies

In [2]:
import json
import re
from neo4j import GraphDatabase
import google.genai as genai          # new SDK — google-genai v1.72
from google.genai import types
import os
from dotenv import load_dotenv
from pathlib import Path


ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
ENV_PATH = ROOT / '.env'

# Load the file from that specific path
load_dotenv(dotenv_path=ENV_PATH)

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')   # <-- update before running
#genai.configure(api_key=GEMINI_API_KEY)


# ── Force v1 stable endpoint to avoid v1beta model deprecations ───────────
# "gemini-1.5-flash" (bare) returns 404 on the default v1beta endpoint.
# Forcing api_version="v1" makes all stable model IDs work correctly.
client = genai.Client(
    api_key      = GEMINI_API_KEY,
    http_options = types.HttpOptions(api_version="v1")
)

# ── Model names (stable, verified for google-genai v1.72 + v1 endpoint) ──
FLASH_MODEL = "gemini-2.0-flash"   # fast, cheap — for classification/routing
PRO_MODEL   = "gemini-1.5-pro"     # long-context — for final generation in NB04

# ── Neo4j ─────────────────────────────────────────────────────────────────
NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update before running

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print(f"Gemini client ready  — Flash: {FLASH_MODEL}")
print(f"Neo4j driver ready   — {NEO4J_URI}")

Gemini client ready  — Flash: gemini-2.0-flash
Neo4j driver ready   — bolt://localhost:7687


## 3.2 Define 5 intent types mapped to feature factors

In [3]:
# Intent types derived strictly from the 11-factor feature set.
# No intent types exist outside these 5.

INTENT_TYPES = {
    "PRICE_ESTIMATION": {
        "description": "User wants a price estimate or valuation",
        "factors": ["resale_price","level_mid","lease_remaining_years",
                    "floor_area_sqm","room_count","all core features"],
    },
    "NEIGHBOURHOOD": {
        "description": "User wants to know about nearby amenities",
        "factors": ["dist_to_mrt_m","dist_to_highway_m","dist_to_foodcourt_m",
                    "mall_count_3km","mall_weighted_access_3km"],
    },
    "SCHOOL_CATCHMENT": {
        "description": "User wants school proximity or quality information",
        "factors": ["dist_to_nearest_school_m","school_count_1km",
                    "primary_school_quality_1km_weighted",
                    "primary_school_top_quality_1km","primary_school_count_1km"],
    },
    "INVESTMENT_TEMPORAL": {
        "description": "User wants historical price trends or appreciation",
        "factors": ["transaction_year","resale_price","town"],
    },
    "LEASE_ADVISORY": {
        "description": "User wants guidance on lease remaining years",
        "factors": ["lease_remaining_years","resale_price"],
    },
}

VALID_TOWNS = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN"
]
VALID_FLAT_TYPES = ["1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"]

print(f"Intent types defined: {list(INTENT_TYPES.keys())}")
print(f"Valid towns: {len(VALID_TOWNS)}, Valid flat types: {len(VALID_FLAT_TYPES)}")

Intent types defined: ['PRICE_ESTIMATION', 'NEIGHBOURHOOD', 'SCHOOL_CATCHMENT', 'INVESTMENT_TEMPORAL', 'LEASE_ADVISORY']
Valid towns: 26, Valid flat types: 7


## 3.3 Gemini Flash — intent classifier prompt

In [4]:
CLASSIFIER_SYSTEM = """You are a query classifier for an HDB flat database system.

Classify the user query into EXACTLY ONE of these intent types:
- PRICE_ESTIMATION: user wants a price estimate or valuation
- NEIGHBOURHOOD: user wants nearby amenities (MRT, malls, food courts, highways)
- SCHOOL_CATCHMENT: user wants school proximity or quality
- INVESTMENT_TEMPORAL: user wants historical price trends or town-level appreciation
- LEASE_ADVISORY: user wants guidance related to lease remaining years

Also extract these slots if mentioned (null if not mentioned):
- town: must be one of the 26 valid HDB towns (uppercase)
- flat_type: must be one of [1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION]
- room_count: integer 1-6
- budget_sgd_min: integer
- budget_sgd_max: integer
- year_min: integer (2015-2026)
- year_max: integer (2015-2026)
- lease_years_max: integer

Respond ONLY with valid JSON. No explanation, no markdown fences.
Format: {"intent": "...", "slots": {"town": null, "flat_type": null, ...}}
"""


def classify_query(user_query: str) -> dict:
    """
    Classify user query using Gemini Flash.
    Old pattern: flash.generate_content(contents=[{...role/parts dict...}])
    New pattern: client.models.generate_content(model=MODEL, contents=str)
    """
    prompt   = CLASSIFIER_SYSTEM + "\n\nQuery: " + user_query
    response = client.models.generate_content(
        model    = FLASH_MODEL,
        contents = prompt
    )
    raw = response.text.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)


# ── Test all 5 intent types ───────────────────────────────────────────────
test_queries = [
    "How much is a 4-room flat in Bishan worth?",
    "What amenities are near Ang Mo Kio flats?",
    "Which areas have the best primary schools under $700K?",
    "Which town had the best price growth from 2020 to 2025?",
    "Should I buy a flat with only 55 years of lease left?",
]

for q in test_queries:
    result = classify_query(q)
    print(f"Q: {q[:55]}")
    print(f"   Intent: {result['intent']} | Slots: {result['slots']}\n")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 49.335020057s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

## 3.4 Cypher template library — one per intent

In [5]:
# All templates use ONLY node properties and relationship types
# defined in Notebook 1. Property names match feature_metadata_20260403.json.

CYPHER_TEMPLATES = {

    "PRICE_ESTIMATION": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
  AND ($room_count IS NULL OR f.room_count = $room_count)
  AND ($budget_sgd_min IS NULL OR f.resale_price >= $budget_sgd_min)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
RETURN
  percentileCont(f.resale_price, 0.5)  AS median_price,
  avg(f.resale_price)                  AS avg_price,
  stDev(f.resale_price)                AS std_price,
  min(f.resale_price)                  AS min_price,
  max(f.resale_price)                  AS max_price,
  count(f)                             AS tx_count,
  avg(f.lease_remaining_years)         AS avg_lease,
  avg(f.floor_area_sqm)               AS avg_area,
  min(f.transaction_year)              AS year_min,
  max(f.transaction_year)              AS year_max
""",

    "NEIGHBOURHOOD": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
RETURN
  t.name                                      AS town,
  avg(f.dist_to_mrt_m)                       AS avg_mrt_dist_m,
  avg(f.dist_to_foodcourt_m)                 AS avg_foodcourt_dist_m,
  avg(f.dist_to_nearest_mall_m)              AS avg_mall_dist_m,
  avg(f.mall_count_3km)                      AS avg_mall_count_3km,
  avg(f.mall_weighted_access_3km)            AS avg_mall_access_score,
  avg(f.dist_to_highway_m)                   AS avg_highway_dist_m,
  count(f)                                   AS flat_count
""",

    "SCHOOL_CATCHMENT": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
RETURN
  t.name                                              AS town,
  avg(f.primary_school_quality_1km_weighted)         AS avg_school_quality,
  avg(f.primary_school_top_quality_1km)              AS avg_top_school_quality,
  avg(f.school_count_1km)                            AS avg_schools_in_1km,
  avg(f.dist_to_nearest_school_m)                    AS avg_school_dist_m,
  avg(f.resale_price)                                AS avg_price,
  count(f)                                           AS flat_count
ORDER BY avg_school_quality DESC
""",

    "INVESTMENT_TEMPORAL": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($year_min IS NULL OR f.transaction_year >= $year_min)
  AND ($year_max IS NULL OR f.transaction_year <= $year_max)
RETURN
  t.name                        AS town,
  f.transaction_year            AS year,
  avg(f.resale_price)           AS avg_price,
  percentileCont(f.resale_price, 0.5) AS median_price,
  count(f)                      AS tx_count
ORDER BY t.name, f.transaction_year
""",

    "LEASE_ADVISORY": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($lease_years_max IS NULL OR f.lease_remaining_years <= $lease_years_max)
RETURN
  CASE
    WHEN f.lease_remaining_years < 50 THEN '<50 years'
    WHEN f.lease_remaining_years < 70 THEN '50-69 years'
    ELSE '70+ years'
  END                                AS lease_band,
  avg(f.resale_price)               AS avg_price,
  percentileCont(f.resale_price, 0.5) AS median_price,
  count(f)                          AS tx_count
ORDER BY lease_band
"""
}

print("Cypher templates loaded:", list(CYPHER_TEMPLATES.keys()))

Cypher templates loaded: ['PRICE_ESTIMATION', 'NEIGHBOURHOOD', 'SCHOOL_CATCHMENT', 'INVESTMENT_TEMPORAL', 'LEASE_ADVISORY']


## 3.5 Slot → Cypher parameter mapper

In [6]:
def slots_to_params(slots: dict) -> dict:
    """Map classifier slots to Neo4j query parameters.
    Only passes values that exist in the dataset schema."""
    town = slots.get("town")
    if town and town.upper() not in VALID_TOWNS:
        town = None   # reject invalid town names

    flat_type = slots.get("flat_type")
    if flat_type and flat_type.upper() not in VALID_FLAT_TYPES:
        flat_type = None

    return {
        "town":            town.upper() if town else None,
        "flat_type":       flat_type.upper() if flat_type else None,
        "room_count":      slots.get("room_count"),
        "budget_sgd_min":  slots.get("budget_sgd_min"),
        "budget_sgd_max":  slots.get("budget_sgd_max"),
        "year_min":        slots.get("year_min"),
        "year_max":        slots.get("year_max"),
        "lease_years_max": slots.get("lease_years_max"),
    }

print("Slot mapper defined.")

Slot mapper defined.


## 3.6 End-to-end routing test

In [7]:
def route_and_query(user_query: str) -> dict:
    """Full routing pipeline: classify → build params → run Cypher."""
    classified = classify_query(user_query)
    intent     = classified["intent"]
    params     = slots_to_params(classified["slots"])
    cypher     = CYPHER_TEMPLATES[intent]

    with driver.session() as session:
        result = session.run(cypher, **params)
        records = [dict(r) for r in result]

    return {
        "intent":  intent,
        "slots":   classified["slots"],
        "params":  params,
        "records": records
    }

# Run all 5 intent types
for q in test_queries:
    out = route_and_query(q)
    print(f"Intent: {out['intent']}")
    print(f"  Params: {out['params']}")
    print(f"  Result rows: {len(out['records'])}")
    if out['records']:
        print(f"  First record: {out['records'][0]}")
    print()

driver.close()
print("Notebook 3 complete.")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 22.100384145s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go